## Import

In [1]:
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:1'

%load_ext autoreload
%autoreload 2

## Dataset

In [2]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = '3b'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.4, -0.1, 0, 0.1, 0.4], rhos=[0.5, 0.6, 0.7, 0.8, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.77351285222025


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## Joint training

In [5]:
## Vector copula estimation
from estimators.VCE import VCE

hyperparams.K_components = 5
hyperparams.nde_type = 'VGC'

ret = []
time_spend = []
for i in range(8):
    t0 = time.time()
    
    estimator = VCE(None, None, architecture_critic, hyperparams)
    estimator.to(device)
    estimator.learn(X, Y)

    mi = estimator.MI(X, Y)
    t = time.time()-t0

    print('true MI:', dataset.empirical_mutual_info())
    print('est MI:', estimator.MI(X, Y))
    
    ret.append(mi)
    time_spend.append(t)

K components= 5 copula transform= True
nde type: VGC
finished: t= 0 loss= 3423.41357421875 loss val= 3592.05419921875 best val loss= 3592.05419921875 best t= 0
finished: t= 51 loss= 79.32666015625 loss val= 82.09371948242188 best val loss= 82.09371948242188 best t= 51
finished: t= 102 loss= 66.59589385986328 loss val= 68.26136779785156 best val loss= 68.03883361816406 best t= 101
finished: t= 153 loss= 60.89224624633789 loss val= 64.10789489746094 best val loss= 63.463619232177734 best t= 151
finished: t= 204 loss= 59.72154235839844 loss val= 62.40105438232422 best val loss= 61.66314697265625 best t= 195
finished: t= 255 loss= 57.51288604736328 loss val= 61.30889129638672 best val loss= 61.221656799316406 best t= 239
finished: t= 306 loss= 58.75049591064453 loss val= 61.99585723876953 best val loss= 60.98419952392578 best t= 285
finished: t= 357 loss= 56.1072998046875 loss val= 62.265445709228516 best val loss= 60.60881805419922 best t= 316
finished: t= 408 loss= 55.928627014160156 los

In [6]:
ret = sorted(ret)

print(np.array(ret)[len(ret)//2], np.array(ret).std(), np.array(ret).max(), np.array(ret).min())
print(np.array(time_spend).mean(), np.array(time_spend).std())

8.121109008789062 0.17632221762945935 8.292901039123535 7.69991397857666
479.19677060842514 69.53144681994843
